In [ ]:
import io
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

pd.set_option('display.max_columns', None)

from owslib.wms import WebMapService
from owslib.wmts import WebMapTileService
from owslib.wfs import WebFeatureService

In [3]:
# Área de estudio: Valle de Aburrá (lon_min, lat_min, lon_max, lat_max)
bbox_aburra = (-75.75, 6.00, -75.30, 6.45)

# Colombia continental
bbox_colombia = (-80.0, -5.0, -66.0, 13.0)

In [ ]:
sgc_rest = 'https://srvags.sgc.gov.co/arcgis/rest/services'

# 1. Hacer la petición GET: URL base + parámetro f=json (pedir la respuesta en JSON)
respuesta = requests.get(sgc_rest, params={'f': 'json'}, timeout=60)

# 2. Revisar la respuesta: código de estado (200 = OK) y la URL completa que se armó
print('Código de estado:', respuesta.status_code)
print('URL:', respuesta.url)

# 3. Convertir el cuerpo de la respuesta (texto JSON) en un diccionario de Python
contenido = respuesta.json()
print('Claves del diccionario:', list(contenido.keys()))

Código de estado: 200
URL: https://srvags.sgc.gov.co/arcgis/rest/services?f=json
Claves del diccionario: ['currentVersion', 'folders', 'services']


In [5]:
# 4. Extraer la lista de carpetas del servidor
carpetas = contenido['folders']
print(carpetas)

['Administracion_IBIC', 'Amenaza_Sismica', 'Amenaza_Volcanica', 'Anomalias_Geoquimica', 'AREAS_HIDRO_GEOQ', 'Atlas_Geologico_2015', 'Atlas_Geologico_Colombiano', 'Atlas_Geoquimico_V2016', 'Atlas_Geoquimico_V2018', 'Cartografia_Basica_IGAC', 'Catalogo_Dataciones_Radiometricas_Colombia', 'catalogo_sismos', 'Elementos_Quimicos_Raros', 'EPIS', 'Estaciones_GNSS', 'Estado_Cartografia_Geologica', 'eval_int', 'Exploracion_Carbones', 'Geofisica', 'Geologia', 'Geological_Map_Colombia_2015', 'Geomecanica_Por_Regiones', 'Geomorfodinamica', 'Geotermia', 'Hidrogeologia_Por_Regiones', 'Inventario_Museo', 'Mapa_Geologico_Colombia', 'Mapa_Geologico_Colombia_2015', 'Mapa_Geologico_Sur_America', 'Mapa_Intensidad_Maxima_Observada_Colombia', 'Mapa_Inventario_Muestra_Litoteca', 'Mapa_Metalogenico_2016', 'Mapa_metalogenico_2018', 'Mapa_Metalogenico_2022', 'Mapa_Metalogenico_Colombia_2002', 'Mapa_Nacional_Amenaza_Mov_Masa_100K', 'Mapa_Potencial_Carbonifero_Colombia', 'Mapa_Sombras_Colombia', 'METG', 'MTER', '

In [7]:
# 2. Hacer la petición pidiendo la respuesta en JSON
respuesta = requests.get('https://srvags.sgc.gov.co/arcgis/rest/services/Mapa_Geologico_Colombia', params={'f': 'json'}, timeout=60)
print('Código de estado:', respuesta.status_code)

# 3. Convertir la respuesta en diccionario
contenido = respuesta.json()

# 4. Extraer la lista de servicios: cada uno es un diccionario con su nombre y su tipo
servicios = contenido['services']
servicios

Código de estado: 200


[{'name': 'Mapa_Geologico_Colombia/Mapa_Geologico_Colombia_V2023',
  'type': 'FeatureServer'},
 {'name': 'Mapa_Geologico_Colombia/Mapa_Geologico_Colombia_V2023',
  'type': 'MapServer'},
 {'name': 'Mapa_Geologico_Colombia/Mapa_Geologico_Colombia',
  'type': 'MapServer'}]

In [18]:
# 2. Pedir la ficha en JSON
respuesta = requests.get('https://srvags.sgc.gov.co/arcgis/rest/services/Mapa_Geologico_Colombia/Mapa_Geologico_Colombia_V2023/MapServer', params={'f': 'json'}, timeout=60)
respuesta.json()  # Convertir la respuesta en diccionario

{'currentVersion': 10.91,
 'serviceDescription': "<div style='text-align:Left;'><div><p><span>El presente servicio visualiza el Atlas Geológico de Colombia a escala 1:500.000 elaborado porel Servicio Geológico Colombiano en el año 2023</span></p></div></div>",
 'mapName': 'Mapa Geológico de Colombia',
 'description': '',
 'copyrightText': 'Servicio Geológico Colombiano',
 'supportsDynamicLayers': True,
 'layers': [{'id': 0,
   'name': 'Anotaciones',
   'parentLayerId': -1,
   'defaultVisibility': True,
   'subLayerIds': [1, 647, 717, 719, 694, 695, 696, 697],
   'minScale': 0,
   'maxScale': 0,
   'type': 'Group Layer'},
  {'id': 1,
   'name': 'Fallas Anot',
   'parentLayerId': 0,
   'defaultVisibility': True,
   'subLayerIds': [646],
   'minScale': 0,
   'maxScale': 0,
   'type': 'Annotation Layer',
   'geometryType': 'esriGeometryPolygon'},
  {'id': 646,
   'name': 'Nombre de la falla',
   'parentLayerId': 1,
   'defaultVisibility': True,
   'subLayerIds': None,
   'minScale': 0,
   

In [ ]:
html = requests.get("https://srvags.sgc.gov.co/arcgis/rest/services/Mapa_Geologico_Colombia/Mapa_Geologico_Colombia_V2023/MapServer", params={'f': 'html'}, timeout=60).text
print(html)

In [24]:
# 3. Conectarse al servicio: OWSLib hace la petición GetCapabilities
sgc = WebMapService('https://srvags.sgc.gov.co/arcgis/services/Mapa_Geologico_Colombia/Mapa_Geologico_Colombia_V2023/MapServer/WMSServer',
    version='1.1.1', timeout=120)

# 4. Revisar la identificación del servicio
print(sgc.identification.type)

WMS


In [25]:
list(sgc.contents)

['0',
 '1',
 '2',
 '3',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '17',
 '18',
 '19',
 '20',
 '21',
 '23',
 '24',
 '25',
 '26',
 '27',
 '29',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '41',
 '43',
 '45',
 '47',
 '49',
 '51',
 '52',
 '54',
 '55',
 '56',
 '57',
 '58',
 '59',
 '60',
 '61',
 '62',
 '63',
 '64',
 '65',
 '66',
 '69',
 '71',
 '73',
 '75',
 '77',
 '79',
 '81',
 '83']